<div style="background:linear-gradient(120deg,#070F24 0%,#0D1A3A 60%,#1B1350 100%);border:1px solid #1F3161;border-radius:16px;padding:22px 24px;color:#E6EEFF;font-family:Inter,Segoe UI,Arial,sans-serif">
<div style="letter-spacing:4px;font-size:12px;color:#22D3EE;font-weight:700">NOVATEL · DATA FACTORY & DIGITAL ANALYTICS FOR ENTERPRISE</div>
<div style="font-size:26px;font-weight:800;margin:6px 0 4px">Performance Optimization Lab</div>
<div style="color:#8EA2CF">Day 3 · Performance Optimization & Scalability · ⏱ 20–25 minutes</div>
<div style="margin-top:14px;font-size:13.5px;line-height:1.6">
<b>Workflow:</b> Baseline → Optimize → Validate → Measure → Decide<br>
<b>Missions:</b> 01 Explore the Data · 02 Run the Experiment · 03 Scalability · 04 Executive Decisions · 05 Recommendation
</div>
<div dir="rtl" style="text-align:right;color:#8EA2CF;margin-top:12px;font-size:14px">
ابدأ من القائمة: <b>Runtime ← Run all</b> ثم ارفع ملف البيانات عند الطلب. كل الأرقام تُقاس فعليًا على جهازك.
</div>
</div>

### How to run this lab
1. Type your name in the **Participant** cell below.
2. Click **Runtime → Run all** (Colab) or **Run → Run All Cells** (Jupyter).
3. When asked, upload **`telecom_revenue_100k.csv`**.
4. Work through each mission: answer the checks, re-run experiments, complete the recommendation form.
5. Download your **results JSON** at the end and import it into **NovaTel_Performance_Command_Center.html**.

> **What is measured:** every timing in this notebook comes from code executing on *your* runtime with `time.perf_counter()`. Nothing is pre-filled. Results differ between machines — that is expected.

In [ ]:
# @title 👤 Participant — type your name, then run all cells
PARTICIPANT_NAME = ""  # @param {type:"string"}
TEAM = ""  # @param {type:"string"}
print("Participant:", PARTICIPANT_NAME or "(not set — you can still run the lab)", "| Team:", TEAM or "-")

In [ ]:
# @title ⚙️ NovaTel Lab Engine — run this cell first (double-click to inspect the code)
# This cell only DEFINES helper functions and styling. Nothing is measured here.
import json, math, platform, re, sys, time, base64, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

try:
    import ipywidgets as widgets
    WIDGETS = True
except ImportError:          # the lab still works without widgets
    WIDGETS = False

try:
    import google.colab      # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------------------------------------------------------- settings
DISCOUNT_FACTOR = 0.95          # the business rule both methods apply
REPEATS = 7                     # timed repetitions per method (median is reported)
WARMUP_RUNS = 1                 # untimed warm-up run per method (caches, first-call overhead)
RTOL, ATOL = 1e-12, 1e-9        # tolerance for np.allclose correctness check
SCALE_SIZES = [10_000, 50_000, 100_000]
NIGHTLY_ROWS = 12_000_000       # hypothetical production volume (scenario only)
SCHEMA = "novatel-lab-results/v1"

LAB = {"self_checks": {}, "experiment_runs": [], "scalability": {}, "scalability_runs": []}

# ---------------------------------------------------------------- visual identity
C = dict(bg="#070F24", panel="#0D1A3A", line="#1F3161", text="#E6EEFF", muted="#8EA2CF",
         a="#A78BFA", b="#22D3EE", blue="#3B82F6", ok="#34D399", warn="#FBBF24", bad="#F87171")
plt.rcParams.update({
    "figure.facecolor": C["bg"], "axes.facecolor": C["panel"], "axes.edgecolor": C["line"],
    "axes.labelcolor": C["muted"], "axes.titlecolor": C["text"], "text.color": C["text"],
    "xtick.color": C["muted"], "ytick.color": C["muted"], "grid.color": C["line"],
    "axes.grid": True, "grid.linewidth": 0.6, "font.size": 10, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

_CSS = f"""
<style>
.nv{{font-family:Inter,Segoe UI,Roboto,Arial,sans-serif;background:{C['bg']};color:{C['text']};
     border:1px solid {C['line']};border-radius:14px;padding:16px 18px;margin:6px 0}}
.nv h3{{margin:0 0 4px;font-size:15px;letter-spacing:.3px}}
.nv .sub{{color:{C['muted']};font-size:12px;margin-bottom:12px}}
.nv .grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px}}
.nv .kpi{{background:{C['panel']};border:1px solid {C['line']};border-radius:12px;padding:12px 14px;
          border-top:3px solid var(--acc,{C['blue']})}}
.nv .kpi .l{{font-size:11px;color:{C['muted']};text-transform:uppercase;letter-spacing:.8px}}
.nv .kpi .v{{font-size:22px;font-weight:700;margin-top:4px;font-variant-numeric:tabular-nums}}
.nv .kpi .s{{font-size:11px;color:{C['muted']};margin-top:2px}}
.nv .note{{border-radius:10px;padding:9px 12px;margin-top:10px;font-size:12.5px;line-height:1.45}}
.nv .warn{{background:rgba(251,191,36,.10);border:1px solid rgba(251,191,36,.45)}}
.nv .info{{background:rgba(59,130,246,.10);border:1px solid rgba(59,130,246,.45)}}
.nv .good{{background:rgba(52,211,153,.10);border:1px solid rgba(52,211,153,.45)}}
.nv .bad{{background:rgba(248,113,113,.10);border:1px solid rgba(248,113,113,.45)}}
.nv .ar{{direction:rtl;text-align:right;color:{C['muted']};font-size:13px;margin-top:6px}}
.nv .bar{{height:26px;border-radius:7px;margin:4px 0 12px;animation:nvgrow 1.6s cubic-bezier(.2,.8,.2,1) both}}
.nv .blbl{{display:flex;justify-content:space-between;font-size:12.5px}}
.nv .tag{{display:inline-block;font-size:10.5px;font-weight:700;letter-spacing:.8px;padding:2px 8px;
          border-radius:20px;margin-left:6px;vertical-align:middle}}
@keyframes nvgrow{{from{{width:0}}}}
</style>"""


def show(html):
    display(HTML(_CSS + html))


def kpis(title, items, sub=""):
    """items: list of (label, value, small_text, accent_colour)."""
    cards = "".join(
        f'<div class="kpi" style="--acc:{acc}"><div class="l">{l}</div><div class="v">{v}</div>'
        f'<div class="s">{s}</div></div>' for l, v, s, acc in items)
    return f'<div class="nv"><h3>{title}</h3><div class="sub">{sub}</div><div class="grid">{cards}</div></div>'


def note(text, kind="info"):
    return f'<div class="nv" style="padding:4px 14px"><div class="note {kind}">{text}</div></div>'


def safe_div(a, b):
    """Division that never raises: returns NaN when the denominator is zero."""
    return a / b if b else float("nan")


def fmt_num(x, digits=0):
    if x is None or (isinstance(x, float) and not math.isfinite(x)):
        return "n/a"
    return f"{x:,.{digits}f}"


# ---------------------------------------------------------------- the two methods (unchanged logic)
def method_a_row_by_row(revenue):
    """Method A — baseline: Python loop, one record at a time."""
    discounted = []
    for value in revenue:
        discounted.append(value * DISCOUNT_FACTOR)
    return discounted


def method_b_vectorized(revenue):
    """Method B — optimized: one Pandas/NumPy operation on the whole column."""
    return (revenue * DISCOUNT_FACTOR).to_numpy()


def run_benchmark(revenue, repeats=REPEATS, warmup=WARMUP_RUNS):
    """Times ONLY the computation (not CSV loading, charts or the wider pipeline).
    Returns (metrics_dict, row_result_array, vector_result_array)."""
    for _ in range(warmup):                       # untimed warm-up
        method_a_row_by_row(revenue)
        method_b_vectorized(revenue)
    row_times, vec_times = [], []
    row_result = vec_result = None
    for _ in range(repeats):                      # alternate methods, same input, same order as original lab
        t0 = time.perf_counter()
        discounted = method_a_row_by_row(revenue)
        row_times.append(time.perf_counter() - t0)
        row_result = np.array(discounted)

        t0 = time.perf_counter()
        vec_result = method_b_vectorized(revenue)
        vec_times.append(time.perf_counter() - t0)

    passed = bool(np.allclose(row_result, vec_result, rtol=RTOL, atol=ATOL))
    max_diff = float(np.max(np.abs(row_result - vec_result))) if len(row_result) else 0.0
    row_s, vec_s = float(np.median(row_times)), float(np.median(vec_times))
    n = int(len(revenue))

    def spread(ts):  # coefficient of variation around the median, in %
        m = float(np.median(ts))
        return float(np.std(ts) / m * 100) if m else float("nan")

    metrics = {
        "records": n, "repeats": repeats, "warmup_runs": warmup,
        "operation": f"revenue * {DISCOUNT_FACTOR}",
        "row_times_ms": [t * 1000 for t in row_times],
        "vector_times_ms": [t * 1000 for t in vec_times],
        "row_median_ms": row_s * 1000, "vector_median_ms": vec_s * 1000,
        "speedup": safe_div(row_s, vec_s),
        "time_reduction_pct": (1 - vec_s / row_s) * 100 if row_s > 0 else float("nan"),
        "row_throughput": safe_div(n, row_s), "vector_throughput": safe_div(n, vec_s),
        "row_cv_pct": spread(row_times), "vector_cv_pct": spread(vec_times),
        "correctness": {"passed": passed, "method": "np.allclose", "rtol": RTOL, "atol": ATOL,
                        "max_abs_diff": max_diff},
        "measured_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    }
    return metrics, row_result, vec_result


def variability_notes(m, history=None):
    notes = []
    if max(m["row_cv_pct"], m["vector_cv_pct"]) > 15:
        notes.append(f"⚠️ <b>High run-to-run variation</b> (spread: Method A {m['row_cv_pct']:.0f}%, "
                     f"Method B {m['vector_cv_pct']:.0f}%). Other processes on the machine affect timing — "
                     "re-run and treat the speedup as approximate.")
    if m["vector_median_ms"] < 1:
        notes.append(f"⏱️ <b>Very short runtime</b>: Method B took {m['vector_median_ms']:.3f} ms. At this scale, "
                     "timer resolution and function-call overhead are a large share of the measurement, so the "
                     "speedup ratio is sensitive to noise. The warm-up run and 7-run median reduce, but do not "
                     "remove, this effect.")
    if history and len(history) >= 2:
        sp = [h["speedup"] for h in history if h["speedup"] == h["speedup"]]
        if sp and (max(sp) - min(sp)) / min(sp) > 0.25:
            notes.append(f"🔁 <b>Results varied between your runs</b>: speedup ranged from {min(sp):.1f}× to "
                         f"{max(sp):.1f}× across {len(sp)} runs. Report a range or the median, not the best run.")
    return notes


def compare_bars(m, title="Before vs After — median execution time"):
    a, b = m["row_median_ms"], m["vector_median_ms"]
    wb = max(b / a * 100, 0.8) if a else 0.8
    return (f'<div class="nv"><h3>{title}</h3><div class="sub">Bar length is proportional to measured time '
            f'({m["records"]:,} records, median of {m["repeats"]} runs). Shorter is faster.</div>'
            f'<div class="blbl"><span>Method A · Row-by-row (baseline)</span><b>{a:.3f} ms</b></div>'
            f'<div class="bar" style="width:100%;background:linear-gradient(90deg,{C["a"]},#7C3AED)"></div>'
            f'<div class="blbl"><span>Method B · Vectorized (optimized)</span><b>{b:.3f} ms</b></div>'
            f'<div class="bar" style="width:{wb:.2f}%;background:linear-gradient(90deg,{C["b"]},{C["blue"]})"></div>'
            f'<div class="sub" style="margin:0">Speedup <b style="color:{C["b"]}">{m["speedup"]:.1f}×</b> · '
            f'time reduction <b style="color:{C["b"]}">{m["time_reduction_pct"]:.1f}%</b></div></div>')


# ---------------------------------------------------------------- data loading & profiling
def locate_dataset(name="telecom_revenue_100k.csv"):
    for p in [Path(name), Path("/content") / name, Path("data") / name]:
        if p.exists():
            return p
    if IN_COLAB:
        from google.colab import files
        print(f"📤 Upload {name} using the button below")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file uploaded — re-run this cell and choose the CSV.")
        return Path(next(iter(uploaded)))
    raise FileNotFoundError(f"{name} not found. Put it in the same folder as this notebook.")


def profile_dataset(df, file_name):
    missing = df.isna().sum()
    dup_ids = int(df["transaction_id"].duplicated().sum()) if "transaction_id" in df else None
    neg = int((df["revenue"] < 0).sum())
    checks = [
        {"check": "Missing values (all columns)", "result": int(missing.sum()), "pass": int(missing.sum()) == 0},
        {"check": "Duplicate transaction IDs", "result": dup_ids, "pass": dup_ids in (0, None)},
        {"check": "Negative revenue values", "result": neg, "pass": neg == 0},
        {"check": "Revenue column is numeric", "result": str(df["revenue"].dtype),
         "pass": bool(pd.api.types.is_numeric_dtype(df["revenue"]))},
    ]
    dates = pd.to_datetime(df["transaction_date"], errors="coerce") if "transaction_date" in df else None
    return {
        "file": file_name, "rows": int(len(df)), "columns": [str(c) for c in df.columns],
        "dtypes": {str(c): str(t) for c, t in df.dtypes.items()},
        "missing_by_column": {str(c): int(v) for c, v in missing.items()},
        "missing_values_total": int(missing.sum()), "duplicate_transaction_ids": dup_ids,
        "total_revenue": float(df["revenue"].sum()), "revenue_min": float(df["revenue"].min()),
        "revenue_mean": float(df["revenue"].mean()), "revenue_max": float(df["revenue"].max()),
        "unique_customers": int(df["customer_id"].nunique()) if "customer_id" in df else None,
        "regions": {str(k): int(v) for k, v in df["region"].value_counts().items()} if "region" in df else {},
        "service_types": {str(k): int(v) for k, v in df["service_type"].value_counts().items()} if "service_type" in df else {},
        "date_min": str(dates.min().date()) if dates is not None else None,
        "date_max": str(dates.max().date()) if dates is not None else None,
        "quality_checks": checks,
        "quality_status": "PASS" if all(c["pass"] for c in checks) else "REVIEW",
    }


# ---------------------------------------------------------------- knowledge checks (self-check in notebook)
def knowledge_check(qid, question, options, correct, explanation, hint="Re-read the evidence above and try again."):
    """Interactive multiple-choice check. Retries are always allowed."""
    LAB["self_checks"].setdefault(qid, {"question": question, "attempts": 0, "correct": False, "answer": None})
    letters = "ABCDEFG"
    labels = [f"{letters[i]}. {o}" for i, o in enumerate(options)]
    if not WIDGETS:
        print(question)
        for l in labels:
            print("  ", l)
        print(f"\n➡️ Answer with: answer('{qid}', 'B')   (replace B with your letter)")
        LAB["self_checks"][qid].update(options=options, correct_idx=correct, explanation=explanation, hint=hint)
        return
    q = widgets.HTML(f"<div style='font-size:14px;font-weight:600;margin:6px 0'>{question}</div>")
    rb = widgets.RadioButtons(options=labels, value=None, layout=widgets.Layout(width="auto"))
    btn = widgets.Button(description="Check answer", button_style="info", icon="check")
    out = widgets.Output()

    def on_click(_):
        with out:
            out.clear_output()
            if rb.value is None:
                show(note("Select an option first.", "warn"))
                return
            rec = LAB["self_checks"][qid]
            rec["attempts"] += 1
            idx = labels.index(rb.value)
            rec["answer"] = letters[idx]
            if idx == correct:
                rec["correct"] = True
                show(note(f"✅ <b>Correct</b> (attempt {rec['attempts']}). {explanation}", "good"))
            else:
                show(note(f"❌ Not quite. {hint} You can retry as many times as you need.", "bad"))

    btn.on_click(on_click)
    display(widgets.VBox([q, rb, btn, out]))
    return btn, rb


def answer(qid, letter):
    """Fallback for environments without ipywidgets."""
    rec = LAB["self_checks"][qid]
    rec["attempts"] += 1
    idx = "ABCDEFG".index(letter.strip().upper())
    rec["answer"] = letter.upper()
    if idx == rec["correct_idx"]:
        rec["correct"] = True
        print("✅ Correct.", rec["explanation"])
    else:
        print("❌ Not quite.", rec["hint"])


# ---------------------------------------------------------------- export
def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        o = float(o)
        return float(f"{o:.10g}") if math.isfinite(o) else None
    if isinstance(o, np.bool_):
        return bool(o)
    return o


def build_results(participant_name="", team=""):
    return _clean({
        "schema": SCHEMA,
        "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "source": "NovaTel_Interactive_Performance_Lab.ipynb",
        "participant": {"name": participant_name.strip(), "team": team.strip()},
        "environment": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                        "platform": platform.platform(terse=True), "machine": platform.machine(),
                        "colab": IN_COLAB},
        "dataset": LAB.get("dataset"),
        "benchmark": LAB.get("experiment"),
        "benchmark_history": [{"speedup": r["speedup"], "row_median_ms": r["row_median_ms"],
                               "vector_median_ms": r["vector_median_ms"], "measured_at": r["measured_at"]}
                              for r in LAB["experiment_runs"]],
        "scalability": [LAB["scalability"][k] for k in sorted(LAB["scalability"])],
        "simulation": LAB.get("simulation"),
        "notebook_self_checks": {k: {kk: v.get(kk) for kk in ("attempts", "correct", "answer")}
                                 for k, v in LAB["self_checks"].items()},
        "notebook_recommendation": LAB.get("recommendation"),
    })


def results_filename(name):
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_") or "participant"
    return f"novatel_results_{slug}.json"


def save_results(participant_name="", team=""):
    data = build_results(participant_name, team)
    path = Path(results_filename(participant_name))
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    return path, data


def download_link_html(path):
    b64 = base64.b64encode(Path(path).read_bytes()).decode()
    return (f'<div class="nv"><h3>📦 Results file ready: {Path(path).name}</h3>'
            f'<div class="sub">Import this file into <b>NovaTel_Performance_Command_Center.html</b>. '
            f'If the button does not download in your browser, run the “Download results” cell below.</div>'
            f'<a download="{Path(path).name}" href="data:application/json;base64,{b64}" '
            f'style="display:inline-block;background:linear-gradient(90deg,{C["blue"]},{C["b"]});color:#04122B;'
            f'font-weight:700;padding:9px 16px;border-radius:9px;text-decoration:none">⬇ Download JSON</a></div>')


print("✅ Lab engine ready.",
      f"Python {platform.python_version()} · pandas {pd.__version__} · numpy {np.__version__}",
      "· widgets:", "on" if WIDGETS else "off (text fallback)", "· Colab" if IN_COLAB else "· local Jupyter")

---
## 🛰️ Mission 01 — Explore the Data
**Business context:** NovaTel's billing analytics team applies a **5% loyalty discount** to transaction revenue before reporting. You have a sample of **100,000 synthetic telecom transactions** across five Saudi regions and four service lines.

**Your tasks:** load the dataset · count transactions · identify fields · check missing values · understand what each record represents.

<div dir="rtl" style="text-align:right">المهمة ١: حمّل البيانات وتحقق من جودتها قبل أي قياس للأداء — لا قيمة لقياس سريع على بيانات غير صحيحة.</div>

In [ ]:
# Mission 01 — load and profile the dataset
path = locate_dataset()
df = pd.read_csv(path)

# Quality gates from the original lab
assert "revenue" in df.columns, "The dataset must contain a 'revenue' column"
assert pd.api.types.is_numeric_dtype(df["revenue"]), "'revenue' must be numeric"
assert df["revenue"].notna().all(), "'revenue' contains missing values"

profile = profile_dataset(df, path.name)
LAB["dataset"] = profile
status_col = C["ok"] if profile["quality_status"] == "PASS" else C["warn"]
show(kpis("Dataset overview", [
    ("Total transactions", f"{profile['rows']:,}", f"{len(profile['columns'])} fields", C["blue"]),
    ("Total revenue", f"SAR {profile['total_revenue']/1e6:,.2f}M", f"avg SAR {profile['revenue_mean']:,.2f}", C["b"]),
    ("Regions", len(profile["regions"]), ", ".join(profile["regions"]), C["a"]),
    ("Service types", len(profile["service_types"]), ", ".join(profile["service_types"]), C["a"]),
    ("Data quality", profile["quality_status"], f"{profile['missing_values_total']} missing · "
     f"{profile['duplicate_transaction_ids']} duplicate IDs", status_col),
], sub=f"{profile['file']} · {profile['date_min']} → {profile['date_max']} · {profile['unique_customers']:,} customers"))

print("Fields and data quality:")
display(pd.DataFrame({"dtype": profile["dtypes"], "missing": profile["missing_by_column"],
                      "unique values": df.nunique()}))
print("\nFirst 5 transactions:")
display(df.head())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
pd.Series(profile["regions"]).sort_values().plot.barh(ax=axes[0], color=C["blue"])
axes[0].set_title("Transactions by region"); axes[0].set_xlabel("records")
pd.Series(profile["service_types"]).sort_values().plot.barh(ax=axes[1], color=C["a"])
axes[1].set_title("Transactions by service type"); axes[1].set_xlabel("records")
plt.tight_layout(); plt.show()

### ✅ Knowledge check 01
Answer before continuing. Retries are unlimited (scored attempts happen in the Command Center).

In [ ]:
knowledge_check(
    "m1_q1",
    "Why do we check missing values and data types BEFORE benchmarking performance?",
    ["Missing values make Python run faster, which would distort the benchmark.",
     "A faster method is only useful if it produces correct results on valid input — quality comes first.",
     "Benchmarks can only be run on datasets with exactly 100,000 rows.",
     "Data types do not matter for vectorized operations."],
    correct=1,
    explanation="Performance evidence is meaningless if the input is invalid or the output is wrong. "
                "Validate data, then optimize, then validate the result again.",
    hint="Think about what a manager would ask first: ‘is it fast?’ or ‘is it right?’")

knowledge_check(
    "m1_q2",
    f"Each row in this dataset represents…",
    ["One customer's total yearly revenue.",
     "One region's monthly revenue summary.",
     "One billing transaction (customer, date, region, service, revenue).",
     "One network tower's traffic volume."],
    correct=2,
    explanation="Each row has a unique transaction_id plus customer, date, region, service type and revenue.",
    hint="Look at the column list and the ‘unique values’ count for transaction_id.")

---
## ⚡ Mission 02 — Run the Performance Experiment
Both methods apply **the same business rule**: `discounted_revenue = revenue × 0.95`.

| | Method A — Row-by-Row (baseline) | Method B — Vectorized (optimized) |
|---|---|---|
| How | A Python `for` loop touches each value one at a time | One Pandas/NumPy operation on the whole column |
| Code | `for value in revenue: out.append(value * 0.95)` | `(revenue * 0.95).to_numpy()` |

**Protocol:** 1 untimed warm-up run → **7 timed repetitions** per method → report the **median** → verify results with **`np.allclose`**.
The timer covers **only the computation** — not CSV loading, charting, or the end-to-end pipeline.

<div dir="rtl" style="text-align:right">شغّل الخلية التالية. يمكنك إعادة التجربة بالزر لترى كيف تختلف النتائج بين التشغيلات.</div>

In [ ]:
# Mission 02 — benchmark Method A vs Method B on the full dataset
revenue = df["revenue"]
exp_out = widgets.Output() if WIDGETS else None


def render_experiment(m):
    ok = m["correctness"]["passed"]
    show(kpis(f"Experiment run #{len(LAB['experiment_runs'])} — measured on this runtime", [
        ("Baseline time (A)", f"{m['row_median_ms']:.3f} ms", f"median of {m['repeats']} runs", C["a"]),
        ("Optimized time (B)", f"{m['vector_median_ms']:.3f} ms", f"median of {m['repeats']} runs", C["b"]),
        ("Speedup", f"{m['speedup']:.1f}×", "A time ÷ B time", C["blue"]),
        ("Time reduction", f"{m['time_reduction_pct']:.1f}%", "(1 − B/A) × 100", C["blue"]),
        ("Throughput (B)", f"{m['vector_throughput']/1e6:,.1f}M rec/s", f"A: {m['row_throughput']/1e6:,.2f}M rec/s", C["b"]),
        ("Correctness", "PASS" if ok else "FAIL", f"max |A−B| = {m['correctness']['max_abs_diff']:.2e}",
         C["ok"] if ok else C["bad"]),
    ], sub=f"{m['records']:,} records · operation: {m['operation']} · timer: time.perf_counter()"))
    show(compare_bars(m))
    for n in variability_notes(m, LAB["experiment_runs"]):
        show(note(n, "warn"))

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    vals = [m["row_median_ms"], m["vector_median_ms"]]
    bars = axes[0].bar(["Row-by-row", "Vectorized"], vals, color=[C["a"], C["b"]])
    for bar, val in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.2f} ms", ha="center", va="bottom")
    axes[0].set_ylim(0, max(vals) * 1.2); axes[0].set_ylabel("Median execution time (ms)")
    axes[0].set_title(f"Median time — {m['records']:,} records")
    runs = range(1, m["repeats"] + 1)
    axes[1].plot(runs, m["row_times_ms"], "o-", color=C["a"], label="Row-by-row")
    axes[1].plot(runs, m["vector_times_ms"], "o-", color=C["b"], label="Vectorized")
    axes[1].set_yscale("log"); axes[1].set_xlabel("Repetition"); axes[1].set_ylabel("ms (log scale)")
    axes[1].set_title("Every timed repetition"); axes[1].legend()
    plt.tight_layout(); plt.show()


def run_experiment(_=None):
    m, row_result, vector_result = run_benchmark(revenue)
    assert m["correctness"]["passed"], "Correctness check FAILED — the methods disagree."
    df["discounted_revenue"] = vector_result
    LAB["experiment_runs"].append(m)
    LAB["experiment"] = m
    if WIDGETS:
        with exp_out:
            exp_out.clear_output(wait=True)
            render_experiment(m)
    else:
        render_experiment(m)
    return m


if WIDGETS:
    rerun_btn = widgets.Button(description="Re-run experiment", icon="refresh", button_style="info")
    rerun_btn.on_click(run_experiment)
    display(widgets.HBox([rerun_btn, widgets.HTML(
        "<span style='color:#8EA2CF;font-size:12px;margin-left:8px'>Each click runs a new real benchmark "
        "(1 warm-up + 7 timed runs per method).</span>")]), exp_out)

run_experiment()
display(df[["transaction_id", "revenue", "discounted_revenue"]].head(5))
print("Note: Performance varies by hardware, data types, and workload.")

### 📊 Evidence table
The same metrics as the original lab, computed from your **latest** run.

In [ ]:
m = LAB["experiment"]
summary = pd.DataFrame({
    "Metric": ["Rows processed", "Baseline time (ms)", "Optimized time (ms)", "Speedup (x)",
               "Time reduction (%)", "Baseline throughput (rows/s)", "Optimized throughput (rows/s)",
               "Correctness (np.allclose)"],
    "Measured value": [f"{m['records']:,}", f"{m['row_median_ms']:.3f}", f"{m['vector_median_ms']:.3f}",
                       f"{m['speedup']:.2f}", f"{m['time_reduction_pct']:.2f}", f"{m['row_throughput']:,.0f}",
                       f"{m['vector_throughput']:,.0f}", "PASS" if m["correctness"]["passed"] else "FAIL"],
})
display(summary)

### ✅ Knowledge check 02 — interpret your evidence

In [ ]:
m = LAB["experiment"]
knowledge_check(
    "m2_q1",
    f"Your run shows a speedup of {m['speedup']:.1f}× and a time reduction of {m['time_reduction_pct']:.1f}%. "
    "Which statement is correct?",
    [f"The vectorized method is {m['time_reduction_pct']:.0f}× faster.",
     f"The vectorized method needs about {100 - m['time_reduction_pct']:.1f}% of the baseline time for this computation.",
     "The whole nightly billing pipeline will now finish this much faster.",
     "The speedup proves vectorization is the best choice for every workload."],
    correct=1,
    explanation="Time reduction is a percentage of the baseline time; speedup is a ratio. Both describe this "
                "computation only — not loading, I/O, or the full pipeline.",
    hint="Speedup is a ratio (×); time reduction is a percentage of the baseline.")

knowledge_check(
    "m2_q2",
    "Which piece of evidence shows the optimization did NOT change the business result?",
    ["The vectorized method is faster.",
     "Both methods ran 7 times.",
     "np.allclose(row_result, vector_result) passed with a very small maximum difference.",
     "The bar chart shows a shorter bar for Method B."],
    correct=2,
    explanation="Speed is not correctness. The numerical-equivalence check compares every output value.",
    hint="Which check compares the actual outputs?")

---
## 📈 Mission 03 — Scalability Challenge
Does the advantage stay the same as data grows? The next cell runs a **genuine benchmark** for **10,000 / 50,000 / 100,000** records (first *n* rows, same protocol). Then use the selector to re-run any size and watch the charts update.

<div dir="rtl" style="text-align:right">اختر حجم البيانات واضغط تشغيل — كل اختيار ينفّذ قياسًا حقيقيًا جديدًا.</div>

In [ ]:
# Mission 03 — benchmark several dataset sizes
sizes = [s for s in SCALE_SIZES if s <= len(df)]
scale_out = widgets.Output() if WIDGETS else None


def run_scale(n):
    m, _, _ = run_benchmark(df["revenue"].iloc[:n])
    LAB["scalability"][n] = m
    LAB["scalability_runs"].append({"size": n, "speedup": m["speedup"], "measured_at": m["measured_at"]})
    return m


def render_scale():
    rows = [LAB["scalability"][n] for n in sorted(LAB["scalability"])]
    table = pd.DataFrame({
        "Records": [f"{r['records']:,}" for r in rows],
        "A: row-by-row (ms)": [f"{r['row_median_ms']:.3f}" for r in rows],
        "B: vectorized (ms)": [f"{r['vector_median_ms']:.3f}" for r in rows],
        "Speedup (×)": [f"{r['speedup']:.1f}" for r in rows],
        "A throughput (rec/s)": [f"{r['row_throughput']:,.0f}" for r in rows],
        "B throughput (rec/s)": [f"{r['vector_throughput']:,.0f}" for r in rows],
        "A µs / record": [f"{r['row_median_ms']*1000/r['records']:.4f}" for r in rows],
        "B µs / record": [f"{r['vector_median_ms']*1000/r['records']:.5f}" for r in rows],
        "Correct": ["PASS" if r["correctness"]["passed"] else "FAIL" for r in rows],
    })
    display(table)
    x = [r["records"] for r in rows]
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
    axes[0].plot(x, [r["row_median_ms"] for r in rows], "o-", color=C["a"], label="Row-by-row")
    axes[0].plot(x, [r["vector_median_ms"] for r in rows], "o-", color=C["b"], label="Vectorized")
    axes[0].set_yscale("log"); axes[0].set_title("Processing time vs size"); axes[0].set_ylabel("ms (log)")
    axes[0].legend()
    axes[1].plot(x, [r["row_throughput"] / 1e6 for r in rows], "o-", color=C["a"], label="Row-by-row")
    axes[1].plot(x, [r["vector_throughput"] / 1e6 for r in rows], "o-", color=C["b"], label="Vectorized")
    axes[1].set_yscale("log"); axes[1].set_title("Throughput vs size"); axes[1].set_ylabel("M records/s (log)")
    axes[1].legend()
    axes[2].plot(x, [r["speedup"] for r in rows], "o-", color=C["blue"])
    for xi, r in zip(x, rows):
        axes[2].annotate(f"{r['speedup']:.0f}×", (xi, r["speedup"]), textcoords="offset points", xytext=(0, 7), ha="center")
    axes[2].set_title("Speedup vs size"); axes[2].set_ylabel("× faster"); axes[2].set_ylim(0, max(r["speedup"] for r in rows) * 1.25)
    for ax in axes:
        ax.set_xlabel("records"); ax.set_xticks(x); ax.set_xticklabels([f"{v//1000}k" for v in x])
    plt.suptitle("All points are measured on this runtime (no extrapolation)", color=C["muted"], fontsize=9, y=1.02)
    plt.tight_layout(); plt.show()
    small = min(rows, key=lambda r: r["records"])
    if small["vector_median_ms"] < 1:
        show(note(f"⏱️ At {small['records']:,} records Method B takes only {small['vector_median_ms']:.3f} ms — "
                  "fixed overhead (function calls, memory allocation) is a large part of that, which is why "
                  "speedup often grows with size before levelling off.", "info"))


for n in sizes:          # genuine benchmark for every size
    run_scale(n)

if WIDGETS:
    size_dd = widgets.Dropdown(options=[(f"{s:,} records", s) for s in sizes], value=sizes[0], description="Size:")
    scale_btn = widgets.Button(description="Run benchmark", icon="play", button_style="info")

    def on_scale(_):
        with scale_out:
            scale_out.clear_output(wait=True)
            m = run_scale(size_dd.value)
            show(note(f"New measurement for {size_dd.value:,} records: speedup {m['speedup']:.1f}× "
                      f"(A {m['row_median_ms']:.3f} ms · B {m['vector_median_ms']:.3f} ms).", "good"))
            render_scale()

    scale_btn.on_click(on_scale)
    display(widgets.HBox([size_dd, scale_btn]), scale_out)
    with scale_out:
        render_scale()
else:
    render_scale()

### Why performance does not scale perfectly linearly
- **Fixed overhead:** every call has a constant cost (function dispatch, allocating the output array). On small inputs it is a large share of Method B's time, so its speedup looks smaller.
- **Per-record interpreter cost:** Method A pays Python's per-item cost on every value, so it grows roughly in proportion to the row count.
- **Memory & caches:** as arrays grow beyond CPU caches, memory bandwidth — not arithmetic — becomes the limit.
- **Noise:** sub-millisecond timings are sensitive to background activity, which is why we use warm-up runs and medians.

<div dir="rtl" style="text-align:right">الأداء لا يتضاعف خطيًا دائمًا: التكاليف الثابتة والذاكرة وضوضاء القياس تغيّر النتيجة مع حجم البيانات.</div>

In [ ]:
knowledge_check(
    "m3_q1",
    "Your speedup at 10,000 records differs from the speedup at 100,000 records. The best explanation is…",
    ["The CSV file changed between runs.",
     "Fixed per-call overhead weighs more on small inputs, so the ratio changes with size; noise also matters at sub-millisecond times.",
     "Vectorized code becomes incorrect on larger data.",
     "Python automatically vectorizes loops on small datasets."],
    correct=1,
    explanation="Costs have a fixed part and a per-record part. Their balance shifts with size, so ratios measured at "
                "one size cannot be assumed at another — including 12 million rows.",
    hint="Separate the cost of *starting* an operation from the cost *per record*.")

### 🧪 Enterprise simulation — HYPOTHETICAL (not a measurement)
**Scenario:** NovaTel's billing analytics team processes **12 million transactions nightly**.
The numbers below are a **linear extrapolation** of your measured medians. They exclude I/O, memory pressure, orchestration, network latency, and competing workloads. Use them only to frame the question — **validate at production-like scale before deciding**.

In [ ]:
m = LAB["experiment"]
scale_factor = NIGHTLY_ROWS / m["records"]
est_row_sec = m["row_median_ms"] / 1000 * scale_factor
est_vec_sec = m["vector_median_ms"] / 1000 * scale_factor
# The same extrapolation from each measured size shows how uncertain a linear estimate is
by_size = {n: (r["row_median_ms"] / 1000 * NIGHTLY_ROWS / n, r["vector_median_ms"] / 1000 * NIGHTLY_ROWS / n)
           for n, r in LAB["scalability"].items()}
LAB["simulation"] = {
    "label": "HYPOTHETICAL — linear extrapolation, not a measurement",
    "nightly_rows": NIGHTLY_ROWS, "based_on_records": m["records"],
    "est_row_seconds": est_row_sec, "est_vector_seconds": est_vec_sec,
    "est_time_saved_seconds": est_row_sec - est_vec_sec,
    "range_by_measured_size": {str(n): {"row_seconds": a, "vector_seconds": b} for n, (a, b) in by_size.items()},
}
show(kpis("HYPOTHETICAL · 12M-row nightly compute estimate", [
    ("Illustrative baseline", f"{est_row_sec:,.2f} s", "linear extrapolation", C["a"]),
    ("Illustrative optimized", f"{est_vec_sec:,.3f} s", "linear extrapolation", C["b"]),
    ("Illustrative time saved", f"{est_row_sec - est_vec_sec:,.2f} s", "compute step only", C["warn"]),
], sub="⚠️ Estimate — NOT measured. Excludes I/O, memory pressure, orchestration and concurrency."))
rng_a = [v[0] for v in by_size.values()]; rng_b = [v[1] for v in by_size.values()]
show(note(f"Extrapolating from different measured sizes gives baseline estimates from {min(rng_a):,.2f}s to "
          f"{max(rng_a):,.2f}s and optimized estimates from {min(rng_b):,.3f}s to {max(rng_b):,.3f}s. "
          "The spread itself is evidence that a production-scale test is needed.", "warn"))
print("WARNING: Linear extrapolation is illustrative only; validate at production-like scale.")

---
## 🧭 Mission 04 — Executive Decision Challenge
> *"NovaTel processes millions of telecom billing transactions. Management wants faster analytics without compromising accuracy, reliability, or cost efficiency."*

Answer the three scenarios using **your evidence** from Missions 02–03. These are reasoning questions — there is no trick wording.

<div dir="rtl" style="text-align:right">اتخذ قرارك بناءً على الأدلة التي قستها، لا على الانطباع.</div>

In [ ]:
m = LAB["experiment"]
knowledge_check(
    "m4_q1",
    f"Your evidence: {m['speedup']:.1f}× speedup on {m['records']:,} records, correctness PASS. What do you recommend?",
    ["Deploy the optimized method immediately.",
     "Recommend the optimized method after production-scale validation.",
     "Reject the optimization because it may increase resource consumption."],
    correct=1,
    explanation="The evidence is strong for this operation, but it was measured on a 100k sample in a notebook. "
                "Validate at production scale (volume, memory, end-to-end time, regression tests) before rollout.",
    hint="Is a 100k-row notebook test enough evidence for a 12M-row production system?")

knowledge_check(
    "m4_q2",
    "Method B builds the whole 12M-value result column in memory at once (~96 MB of float64, plus temporaries), "
    "on servers that run several jobs at the same time. What is the most responsible next step?",
    ["Ignore memory — vectorization is always more efficient.",
     "Measure peak memory at production scale; if needed, process in chunks while keeping vectorized operations.",
     "Return to row-by-row because Python loops always use less memory.",
     "Double the server RAM before running any test."],
    correct=1,
    explanation="Memory is a real production risk, but it is measurable and manageable (chunking). Note that "
                "Method A also stores every result — as Python float objects (~32 bytes each), so the loop is "
                "not a low-memory option here.",
    hint="Choose the option that measures the risk rather than assuming it.")

knowledge_check(
    "m4_q3",
    "This discount step is a small part of a 2-hour nightly pipeline dominated by database extraction (I/O). "
    "Leadership asks whether this optimization alone will cut cloud costs substantially.",
    [f"Yes — a {m['speedup']:.0f}× speedup means the pipeline cost drops {m['speedup']:.0f}×.",
     "Not necessarily — the overall gain is limited by this step's share of total runtime; measure end-to-end, "
     "adopt the low-risk change, and target the real bottleneck next.",
     "No — code optimizations never reduce cost.",
     "Yes — and regression tests can be skipped because results matched on 100k rows."],
    correct=1,
    explanation="Overall improvement is bounded by the fraction of time the optimized step takes (Amdahl's law). "
                "Reliability still requires regression testing.",
    hint="If a step takes 1% of the pipeline, how much can speeding it up save overall?")

---
## 📝 Mission 05 — Executive Recommendation
Measured values are filled in automatically from your experiment. Complete the reasoning fields, then click **Generate summary & save results**.

<div dir="rtl" style="text-align:right">اكتب توصيتك بثلاث جمل: النتيجة ← الدليل ← التوصية، مع ذكر قيد واحد.</div>

In [ ]:
# Mission 05 — recommendation form + results export
m = LAB["experiment"]
rec_out = widgets.Output() if WIDGETS else None
SOLUTIONS = ["Vectorized method — after production-scale validation",
             "Vectorized method — deploy immediately",
             "Keep row-by-row method"]


def _sentence(t):
    t = t.strip()
    return t if t.endswith((".", "!", "?")) else t + "."


def executive_summary(rec):
    rec = {k: _sentence(v) if isinstance(v, str) else v for k, v in rec.items()}
    return (f"FINDING: On {m['records']:,} telecom records, the vectorized method computed the 5% discount in "
            f"{m['vector_median_ms']:.3f} ms versus {m['row_median_ms']:.3f} ms row-by-row (median of {m['repeats']} runs) — "
            f"a {m['speedup']:.1f}× speedup and {m['time_reduction_pct']:.1f}% time reduction, with throughput rising from "
            f"{m['row_throughput']:,.0f} to {m['vector_throughput']:,.0f} records/s. "
            f"EVIDENCE: Outputs were numerically equivalent (np.allclose PASS, max difference "
            f"{m['correctness']['max_abs_diff']:.1e}). "
            f"RECOMMENDATION: {rec['selected_solution']} {rec['final_recommendation']} "
            f"RISK: {rec['operational_risk']} "
            f"NEXT VALIDATION: {rec['next_validation_test']} "
            f"LIMITATION: These timings cover the computation only on this runtime; a 12M-row estimate is hypothetical.")


def save_recommendation(rec):
    rec = {**rec,
           "baseline_ms": m["row_median_ms"], "optimized_ms": m["vector_median_ms"],
           "speedup_x": m["speedup"], "time_reduction_pct": m["time_reduction_pct"],
           "correctness_evidence": f"np.allclose PASS (rtol={RTOL}, atol={ATOL}, max diff {m['correctness']['max_abs_diff']:.1e})"}
    rec["executive_summary"] = executive_summary(rec)
    LAB["recommendation"] = rec
    path, _ = save_results(PARTICIPANT_NAME, TEAM)
    show(f'<div class="nv"><h3>Executive summary</h3><div style="line-height:1.6;font-size:13.5px">'
         f'{rec["executive_summary"]}</div></div>')
    show(download_link_html(path))
    return path


if WIDGETS:
    style = {"description_width": "170px"}
    lay = widgets.Layout(width="95%")
    f_sol = widgets.Dropdown(options=SOLUTIONS, description="Selected solution", style=style, layout=lay)
    f_risk = widgets.Textarea(placeholder="e.g. Peak memory at 12M rows; behaviour with nulls or new data types",
                              description="Operational risk", style=style, layout=lay)
    f_test = widgets.Textarea(placeholder="e.g. Run on a 12M-row staging copy; measure end-to-end time, peak memory and compare totals",
                              description="Next validation test", style=style, layout=lay)
    f_final = widgets.Textarea(placeholder="Your recommendation in 1–2 sentences", description="Final recommendation",
                               style=style, layout=lay)
    gen_btn = widgets.Button(description="Generate summary & save results", icon="file", button_style="success",
                             layout=widgets.Layout(width="300px"))
    measured = widgets.HTML(
        f"<div style='font-size:13px;margin:4px 0 8px'>Baseline <b>{m['row_median_ms']:.3f} ms</b> · "
        f"Optimized <b>{m['vector_median_ms']:.3f} ms</b> · Improvement <b>{m['speedup']:.1f}× / "
        f"{m['time_reduction_pct']:.1f}%</b> · Correctness <b>{'PASS' if m['correctness']['passed'] else 'FAIL'}</b></div>")

    def on_generate(_):
        with rec_out:
            rec_out.clear_output()
            missing = [n for n, w in [("Operational risk", f_risk), ("Next validation test", f_test),
                                      ("Final recommendation", f_final)] if len(w.value.strip()) < 10]
            if missing:
                show(note("Please complete: " + ", ".join(missing) + " (at least a short sentence each).", "warn"))
                return
            save_recommendation({"selected_solution": f_sol.value, "operational_risk": f_risk.value.strip(),
                                 "next_validation_test": f_test.value.strip(),
                                 "final_recommendation": f_final.value.strip()})

    gen_btn.on_click(on_generate)
    display(widgets.VBox([measured, f_sol, f_risk, f_test, f_final, gen_btn, rec_out]))
else:
    print("Widgets unavailable: fill in the dictionary below, then run save_recommendation(my_rec)")
    my_rec = {"selected_solution": SOLUTIONS[0], "operational_risk": "", "next_validation_test": "",
              "final_recommendation": ""}

# Always save a measurement-only results file so nothing is lost
_path, _ = save_results(PARTICIPANT_NAME, TEAM)
print(f"💾 Measurements saved to {_path} (re-saved when you generate the summary).")

In [ ]:
# @title ⬇️ Download results — run this cell after completing Mission 05
_path, _data = save_results(PARTICIPANT_NAME, TEAM)
print(f"Saved {_path} · benchmark: {'yes' if _data['benchmark'] else 'no'} · "
      f"scalability sizes: {len(_data['scalability'])} · recommendation: {'yes' if _data['notebook_recommendation'] else 'not yet'}")
if IN_COLAB:
    from google.colab import files
    files.download(str(_path))
else:
    show(download_link_html(_path))

---
## 🏁 Next step — open the Command Center
1. Open **`NovaTel_Performance_Command_Center.html`** in Chrome or Edge.
2. Click **Import results** and choose your `novatel_results_<name>.json`.
3. Complete the scored missions and export your submission for the trainer.

<div dir="rtl" style="text-align:right">افتح لوحة القيادة واستورد ملف النتائج لإكمال المهام وحساب درجتك النهائية.</div>

### Trainer assessment rubric (original, 10 points)
| Criterion | Points |
|---|---:|
| Reports baseline and optimized measurements accurately | 2 |
| Calculates speedup and percentage reduction correctly | 2 |
| Demonstrates numeric-equivalence check | 2 |
| Justifies recommendation with evidence | 2 |
| Identifies production limitations and a validation test | 2 |

**Trainer note:** A correct recommendation is conditional: vectorization is preferable *for this specific column operation under the tested conditions*, not automatically for every pipeline workload.